In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1,3,4"

In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from datasets import load_dataset, Dataset
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
import logging
from dataclasses import dataclass, field
import os
import random
import torch
from datasets import load_dataset
from tqdm import tqdm
#from trl import  TrlParser
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    HfArgumentParser,
    BitsAndBytesConfig,
        set_seed,

)
#from trl import setup_chat_format
from peft import LoraConfig


from trl import (
   SFTTrainer)
tqdm.pandas()
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/home/srmist35/miniconda3/envs/gemma/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
DATASET_NAME = "open-r1/codeforces-cots"
MODEL_NAME = "microsoft/Phi-4-mini-instruct"
my_token = ""
token = ""

In [4]:
import pandas as pd
clean_df = pd.read_csv('clean_cf.csv')
clean_df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'clean_cf.csv'

In [ ]:
train_data = Dataset.from_pandas(clean_df)

In [6]:
#system_message = """You are a LLM coding and problem solving assistant, answer the given problem in code, give the function to solve and the main driver program - together."""
# def create_conversation(record):
#     sample = {"text": [
        
#         {"role": "user", "content": f"""{record["prompt"]}"""},
#         {"role" : "assistant", "content": f"""{record["generation"]}"""}
#     ]}
#     return sample


# train_data = train_data.map(create_conversation, batched=False)



#train_data.to_json("data/train_dataset.json", orient="records", force_ascii=False)


In [7]:
model_id = "meta-llama/Llama-3.2-3B-Instruct"
use_bf16 = True

In [8]:
max_seq_length=1024

In [9]:
import logging
from dataclasses import dataclass, field
import os

import random
import torch
from datasets import load_dataset
from tqdm import tqdm
#from trl.commands.cli_utils import  TrlParser
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    HfArgumentParser,
    BitsAndBytesConfig,
        set_seed,

)
from trl import setup_chat_format
from peft import LoraConfig


from trl import (
   SFTTrainer)


tqdm.pandas()

# @dataclass
# class ScriptArguments:
#     dataset_path: str = field(
#         default=None,
#         metadata={
#             "help": "Path to the dataset"
#         },
#     )
#     model_id: str = field(
#         default=None, metadata={"help": "Model ID to use for SFT training"}
#     )
#     max_seq_length: int = field(
#         default=512, metadata={"help": "The maximum sequence length for SFT Trainer"}
#     )
#     use_qlora: bool = field(default=False, metadata={"help": "Whether to use QLORA"})
#     merge_adapters: bool = field(
#         metadata={"help": "Whether to merge weights for LoRA."},
#         default=False,
#     )



In [10]:
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True,token=token,device_map="auto")
tokenizer.pad_token = tokenizer.eos_token
#tokenizer.chat_template = LLAMA_3_CHAT_TEMPLATEtorch.backends.cuda.matmul.allow_tf32 = True


In [11]:
# def template_dataset(examples):
#     return{"text":  tokenizer.apply_chat_template(examples["text"], tokenize=False)}
    
# train_data = train_data.map(template_dataset, remove_columns=["text"])


In [12]:
# type(train_data)
# updated_train = train_data.to_pandas()
# updated_train.head()


In [13]:
torch_dtype = torch.float16 
#quant_storage_dtype = torch.float16


print(f"Using QLoRA - {torch_dtype}")
quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch_dtype,
            
    )

        


Using QLoRA - torch.float16


In [14]:
import transformers
from trl import SFTConfig

from transformers import TrainingArguments
# training_args = TrainingArguments(
#       per_device_train_batch_size=4,
#       num_train_epochs=3,
#       learning_rate=2e-4,
#       fp16=True,
#       bf16=False,
#       save_steps=20,
#       output_dir = "./",
#       #packing = False,
#     #     dataset_kwargs={
#     #    "add_special_tokens": False,  # We template with special tokens
#     #    "append_concat_token": False,  # No need to add additional separator token
#     # },
#       dataloader_num_workers=2,
#       # dataset_text_field='messages',
#       optim="paged_adamw_8bit",
#       lr_scheduler_type="cosine",
#       warmup_ratio=0.05,
#       report_to="none",
#     #remove_unused_columns=False
      
#       # use_dora=False,
#       # use_rslora=False
#       # lora_alpha=8,
#       # lora_dropout=0.0
# )
training_args = TrainingArguments(
      per_device_train_batch_size=4,
      num_train_epochs=3,
      learning_rate=2e-4,
      fp16=True,
      bf16=False,
      logging_steps=1,
      save_steps=50,
      output_dir = "./",
      #packing = False,
    #     dataset_kwargs={
    #    "add_special_tokens": False,  # We template with special tokens
    #    "append_concat_token": False,  # No need to add additional separator token
    # },
      #max_seq_length=512,
      dataloader_num_workers=2,
      # dataset_text_field='messages',
      optim="paged_adamw_8bit",
      lr_scheduler_type="cosine",
      warmup_ratio=0.05,
      report_to="none",
    #remove_unused_columns=False
      
      # use_dora=False,
      # use_rslora=False
      # lora_alpha=8,
      # lora_dropout=0.0
)


In [15]:

#training_args.optim = "adamw_bnb_8bit"  
training_args.gradient_checkpointing = True  # Recompute activations
training_args.gradient_accumulation_steps = 2  # Accumulate gradients to reduce VRAM load

In [16]:
training_args.per_device_train_batch_size = 4 # Adjust based on memory
training_args.per_device_eval_batch_size = 8
training_args.dataloader_pin_memory = True
training_args.dataloader_num_workers = 4 
#training_args.fsdp = None  #"full_shard auto_wrap"

In [17]:
# training_args.dataset_text_field = 'messages'

In [17]:
type(train_data)

datasets.arrow_dataset.Dataset

In [18]:
# up_tr_df = train_data.to_pandas()
# up_tr_df.head()
train_data.column_names

['text']

In [17]:
training_args.max_seq_length = 1024

In [20]:
# from datasets import Dataset
# updated_train_data = train_data.to_pandas()
# cols = ['messages']
# updated_train_data = updated_train_data[cols]
# updated_train_data = Dataset.from_pandas(updated_train_data)
# updated_train_data

In [22]:
# mod_train = updated_train_data.train_test_split(test_size=0)
# mod_train
torch.cuda.empty_cache()

In [18]:
from peft import prepare_model_for_kbit_training
from accelerate import PartialState
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
    #device_map={"": PartialState().process_index},
    #device_map={'':torch.cuda.current_device()},
    #device_map={'':torch.cuda.current_device()},
    attn_implementation="sdpa", # use sdpa, alternatively use "flash_attention_2", "sdpa"
    #torch_dtype=torch.float16, # can't use bf16
    #max_memory={0: "30GB", 1:"35GB",2:"35GB", 3:"35GB"},
    use_cache= True,  # this is needed for gradient checkpointing
    token=token
)


model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)
model.train()
    ################
    # PEFT
    ################
    # LoRA config based on QLoRA paper & Sebastian Raschka experiment
peft_config = LoraConfig(
    lora_alpha=8,
    lora_dropout=0.05,
    r=8,
    bias="none",
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)

    ################
    # Training
    ################
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    #dataset_text_field="messages",
    peft_config=peft_config,
    
    tokenizer=tokenizer,
    # dataset_kwargs={
    #    "add_special_tokens": False,  # We template with special tokens
    #    "append_concat_token": False,  # No need to add additional separator token
    # },
    #packing=True,
    
)


  # Enable LoRA gradients


trainer.model.print_trainable_parameters()

    ##########################
    # Train model
    ##########################
    # checkpoint = None
    # if training_args.resume_from_checkpoint is not None:
    #     checkpoint = training_args.resume_from_checkpoint
    # trainer.train(resume_from_checkpoint=checkpoint)


Loading checkpoint shards: 100%|███████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:15<00:00,  7.78s/it]
/tmp/ipykernel_3169047/2820098960.py:37: FutureWarning: `tokenizer` is deprecated and removed starting from version 0.16.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  trainer = SFTTrainer(
Truncating train dataset: 100%|█████████████████████████████████████████████████████████████████████████████| 47780/47780 [09:57<00:00, 80.03 examples/s]
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


trainable params: 2,293,760 || all params: 3,215,043,584 || trainable%: 0.0713


In [20]:
trainer.args.per_device_train_batch_size = 8
trainer.args.per_device_train_batch_size

8

In [23]:
trainer.args.dataloader_num_workers=4
trainer.args.num_train_epochs = 2
trainer.args.num_train_epochs

2

In [24]:
trainer.train()
    

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
TOKENIZERS_PARALLELISM=(true | false)iable 
huggingface/tokenizers: The current process just got forked, afte

Step,Training Loss
1,1.798900
2,1.786800
3,1.741500
4,1.735500
5,1.749600
6,1.755000
7,1.800100
8,1.777200
9,1.782000
10,1.741600


/home/srmist35/miniconda3/envs/gemma/lib/python3.13/site-packages/peft/utils/other.py:716: UserWarning: Unable to fetch remote file due to the following error (ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 19cec0ef-a96a-442a-8aeb-eab80ef40a2c)') - silently ignoring the lookup for the file config.json in meta-llama/Llama-3.2-3B-Instruct.
  warnings.warn(
/home/srmist35/miniconda3/envs/gemma/lib/python3.13/site-packages/peft/utils/save_and_load.py:246: UserWarning: Could not find a config file in meta-llama/Llama-3.2-3B-Instruct - will assume that the vocabulary was not modified.
  warnings.warn(
/home/srmist35/miniconda3/envs/gemma/lib/python3.13/site-packages/peft/utils/other.py:716: UserWarning: Unable to fetch remote file due to the following error (ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 20d6732d-7ed7-411f-942a-95ec946f7bb8)'

TrainOutput(global_step=5972, training_loss=0.8888908718467478, metrics={'train_runtime': 19483.0029, 'train_samples_per_second': 4.905, 'train_steps_per_second': 0.307, 'total_flos': 1.6559448370918195e+18, 'train_loss': 0.8888908718467478})

In [25]:
model.save_pretrained('./codeforces-cots-finetuned')

In [26]:
trainer.model.save_pretrained('llam3-3b-codeforces-coder')

In [ ]:
from accelerate import PartialState
device_map={"": PartialState().process_index}
device_map

In [ ]:
model.save_pretrained()

In [ ]:
model.push_to_hub("Rudrresh/codeforces-cots-llama3",token="")
trainer.tokenizer.push_to_hub("Rudrresh/codeforces-cots-llama3",token="")

model.safetensors: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 3.04G/3.04G [03:29<00:00, 14.5MB/s]
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
tokenizer.json: 100%|███████████████████████████████████████████████████████████████████████████████████████████████| 17.2M/17.2M [00:01<00:00, 10.8MB/s]


CommitInfo(commit_url='https://huggingface.co/Rudrresh/codeforces-cots-llama3/commit/c806e62a0a0af69449afb447be928469baee4249', commit_message='Upload tokenizer', commit_description='', oid='c806e62a0a0af69449afb447be928469baee4249', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Rudrresh/codeforces-cots-llama3', endpoint='https://huggingface.co', repo_type='model', repo_id='Rudrresh/codeforces-cots-llama3'), pr_revision=None, pr_num=None)

In [28]:
model.peft_config

{'default': LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path='meta-llama/Llama-3.2-3B-Instruct', revision=None, inference_mode=False, r=8, target_modules={'v_proj', 'q_proj'}, exclude_modules=None, lora_alpha=8, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', loftq_config={}, eva_config=None, use_dora=False, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False)}

In [ ]:
model.peft_config.push_to_hub("Rudrresh/codeforces-cots-llama3",token="")

AttributeError: 'dict' object has no attribute 'push_to_hub'